# Power BI Usage Intelligence — Forecasting Evaluation Framework

This notebook walks through the complete rolling-origin evaluation pipeline:
load → profile → backtest → summarise → select → refit → produce forecast.

Every piece of modelling logic is imported from `src/`.  No function is
redefined inline.  No fallback or synthetic data is generated here.

**The six production lifecycle steps**

| Step | Module | What it does |
|---|---|---|
| 1 | `backtesting` | Generate rolling-origin train/test splits |
| 2 | `backtest_evaluation` | Evaluate candidates across folds |
| 3 | `model_summary` | Aggregate fold metrics per model |
| 4 | `selection` | Choose one model per report |
| 5 | `production_forecast` | Refit on full history; generate 28-day forecast |
| 6 | `run_forecasting_pipeline` | Save outputs with full lineage |

## Prerequisites

1. Run `notebooks/04_feature_engineering.ipynb` to produce
   `data/processed/mart_report_daily_series.csv`.
2. This notebook reads **only** `mart_report_daily_series.csv`.
   No other data source is accepted.
3. ETS and Auto-ARIMA require `statsmodels` and `pmdarima`.
   Set `USE_FAST_MODELS_ONLY = False` in the configuration cell
   to enable them (significant extra runtime).

In [ ]:
import uuid
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Pipeline utilities ───────────────────────────────────────────────────────
from src.pipelines.run_forecasting_pipeline import (
    STANDARD_DATE_COL,
    STANDARD_REPORT_ID_COL,
    STANDARD_TARGET_COL,
    MIN_DAYS,
    MIN_NONZERO_DAYS,
    MIN_NONZERO_RATIO,
    MIN_TOTAL_VIEWS,
    adapt_to_forecasting_schema,
    build_daily_series_for_all_reports,
    filter_by_data_criteria,
    get_project_root,
    load_canonical_daily_series,
    load_forecast_feature_input,
    run_data_quality_checks,
    validate_forecasting_series_input,
)

# ── Forecasting configuration ─────────────────────────────────────────────────
from src.config.forecasting import (
    BACKTEST_FOLDS,
    BACKTEST_STEP_DAYS,
    FORECAST_HORIZON_DAYS,
    MIN_TRAIN_DAYS,
    MIN_VALID_FOLDS,
    SEASONAL_PERIOD,
)

# ── Rolling-origin splits ─────────────────────────────────────────────────────
from src.models.backtesting import ForecastFold, generate_rolling_splits

# ── Candidate models ──────────────────────────────────────────────────────────
from src.models.candidates import (
    ModelResult,
    forecast_auto_arima,
    forecast_ets,
    forecast_moving_average,
    forecast_naive,
    forecast_seasonal_naive,
)

# ── Fold evaluation ───────────────────────────────────────────────────────────
from src.models.backtest_evaluation import BacktestConfig, evaluate_models_across_folds

# ── Horizon-bucket metrics ────────────────────────────────────────────────────
from src.models.horizon_evaluation import HORIZON_BUCKETS, calculate_horizon_bucket_metrics

# ── Cross-fold model summary ──────────────────────────────────────────────────
from src.models.model_summary import summarise_model_performance

# ── Model selection ───────────────────────────────────────────────────────────
from src.models.selection import (
    MAX_BIAS_RATIO,
    RELATIVE_IMPROVEMENT_TOLERANCE,
    select_models,
)

# ── Production forecast ───────────────────────────────────────────────────────
from src.models.production_forecast import PRODUCTION_FORECAST_COLS, build_production_forecast

In [ ]:
PROJECT_ROOT = get_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR    = PROJECT_ROOT / "outputs"

RUN_TIMESTAMP = pd.Timestamp.now()
RUN_ID = RUN_TIMESTAMP.strftime("%Y%m%d_%H%M%S") + "_" + str(uuid.uuid4())[:8]

# ── Demonstration controls ────────────────────────────────────────────────────
# DEMO_REPORT_ID: pin the per-report visuals to one specific ID.
# Leave as None to auto-select the longest eligible series.
DEMO_REPORT_ID: str | None = None

# USE_FAST_MODELS_ONLY: skips ETS and Auto-ARIMA (which require optional
# packages and add significant runtime).  Set to False to include them.
USE_FAST_MODELS_ONLY: bool = True

# MAX_REPORTS_FOR_SUMMARY: cap on reports evaluated in section 9.
# Increase to process all eligible reports (slower).
MAX_REPORTS_FOR_SUMMARY: int = 5

# BacktestConfig for this notebook run.
DEMO_BACKTEST_CONFIG = BacktestConfig(
    horizon=FORECAST_HORIZON_DAYS,
    n_folds=BACKTEST_FOLDS,
    step=BACKTEST_STEP_DAYS,
    min_train_size=MIN_TRAIN_DAYS,
    seasonal_period=SEASONAL_PERIOD,
)

print(f"RUN_ID:               {RUN_ID}")
print(f"PROJECT_ROOT:         {PROJECT_ROOT.resolve()}")
print(f"FORECAST_HORIZON:     {FORECAST_HORIZON_DAYS} days")
print(f"BACKTEST_FOLDS:       {BACKTEST_FOLDS}")
print(f"SEASONAL_PERIOD:      {SEASONAL_PERIOD} days (weekly)")
print(f"MIN_TRAIN_DAYS:       {MIN_TRAIN_DAYS}")
print(f"MIN_VALID_FOLDS:      {MIN_VALID_FOLDS}")
print(f"USE_FAST_MODELS_ONLY: {USE_FAST_MODELS_ONLY}")

## 1. Evaluation Objective

### 28-day forecast horizon

Each production forecast covers the next **28 days** (four full weeks).
The horizon equals four seasonal periods so that comparisons between models
are not distorted by partial-week boundary effects.

### Rolling-origin backtesting

Models are evaluated using **rolling-origin (expanding-window) backtesting**:

1. Select a *cutoff date*.  Everything before it is the training window.
2. Fit the model on the training window only.
3. Forecast the next 28 days.
4. Compare forecasts against the held-out actuals.
5. Advance the cutoff by `BACKTEST_STEP_DAYS` (= `FORECAST_HORIZON_DAYS`) and repeat.

This produces non-overlapping test windows and prevents any test observation
from appearing in the training set.  Each fold is independent.

### Why baselines are required

Seasonal naive (repeat the last observed weekly pattern) is the minimum
acceptable benchmark.  A model that cannot outperform seasonal naive on
most backtest folds provides no practical value over a zero-implementation
baseline, regardless of its in-sample fit.

The selection policy enforces a **relative improvement tolerance** of
`5%` over seasonal naive before a more complex model is preferred.

## 2. Load Canonical Daily Series

The pipeline accepts **only** `mart_report_daily_series.csv` as its
forecasting input.  The loader raises `FileNotFoundError` if the file
is absent — there is no fallback and no synthetic data generation.

In [ ]:
daily_series_df, report_views_df, active_series_input_file = load_forecast_feature_input(PROJECT_ROOT)

print(f"Loaded:      {active_series_input_file.relative_to(PROJECT_ROOT)}")
print(f"Shape:       {daily_series_df.shape}")
print(f"Reports:     {daily_series_df[STANDARD_REPORT_ID_COL].nunique()}")
print(f"Date range:  "
      f"{daily_series_df[STANDARD_DATE_COL].min().date()} → "
      f"{daily_series_df[STANDARD_DATE_COL].max().date()}")

dq = run_data_quality_checks(daily_series_df)
print("\nData quality checks:")
display(dq)

## 3. Series Profiling

Eligibility gates (applied per report):

| Gate | Threshold | Rationale |
|---|---|---|
| Minimum history | ≥ `MIN_DAYS` days | Enough for seasonal decomposition |
| Non-zero days | ≥ `MIN_NONZERO_DAYS` | Series must have some activity |
| Non-zero ratio | ≥ `MIN_NONZERO_RATIO` | Guards against mostly-dark reports |
| Total views | ≥ `MIN_TOTAL_VIEWS` | Minimum signal threshold |

Reports that fail any gate are excluded from modelling.

In [ ]:
series_dict, name_lookup, provenance = build_daily_series_for_all_reports(report_views_df)
passing_ids, data_diag = filter_by_data_criteria(series_dict, provenance)

if not passing_ids:
    raise RuntimeError(
        "No reports passed eligibility criteria.\n"
        "Ensure mart_report_daily_series.csv contains sufficient history "
        "(run notebooks/04_feature_engineering.ipynb first)."
    )

profile_cols = [
    "report_id", "n_obs", "date_min", "date_max",
    "nonzero_days", "zero_share", "total_views",
    "passes_min_days", "passes_nonzero_ratio", "passes_data_criteria",
]
display_profile = data_diag[[c for c in profile_cols if c in data_diag.columns]]
print(f"Total reports in series: {len(series_dict)}")
print(f"Passing eligibility:     {len(passing_ids)}")
display(
    display_profile
    .sort_values("passes_data_criteria", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

## 4. Rolling-Fold Visualization

The diagram below shows the expanding training windows and fixed-width
28-day test windows for one representative report.  Each fold advances
the cutoff by one horizon length, so test windows never overlap.

In [ ]:
# Select demo report
if DEMO_REPORT_ID is not None and DEMO_REPORT_ID in series_dict:
    demo_rid = DEMO_REPORT_ID
else:
    demo_rid = max(passing_ids, key=lambda r: len(series_dict[r]))

demo_series = series_dict[demo_rid]
demo_name   = name_lookup.get(demo_rid, demo_rid)

print(f"Demo report: {demo_name!r}  ({demo_rid})")
print(f"History:     {demo_series.index.min().date()} → {demo_series.index.max().date()} "
      f"({len(demo_series)} days)")

folds, split_status = generate_rolling_splits(
    demo_series,
    horizon   = DEMO_BACKTEST_CONFIG.horizon,
    n_folds   = DEMO_BACKTEST_CONFIG.n_folds,
    step      = DEMO_BACKTEST_CONFIG.step,
    min_train_size = DEMO_BACKTEST_CONFIG.min_train_size,
)
print(f"Folds generated: {len(folds)}  (ok={split_status['ok']})")

# ── Gantt-style fold window chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, max(3, len(folds) * 1.0 + 1.5)))
_TRAIN_CLR, _TEST_CLR = "steelblue", "tomato"

for fold in folds:
    y = fold.fold_number
    ax.barh(y, (fold.train_end - fold.train_start).days,
            left=fold.train_start, color=_TRAIN_CLR, alpha=0.65, height=0.55)
    ax.barh(y, (fold.test_end - fold.test_start).days,
            left=fold.test_start, color=_TEST_CLR, alpha=0.9, height=0.55)

handles = [
    mpatches.Patch(color=_TRAIN_CLR, alpha=0.65, label="Training window (expanding)"),
    mpatches.Patch(color=_TEST_CLR,  alpha=0.9,  label=f"Test window ({FORECAST_HORIZON_DAYS} days)"),
]
ax.legend(handles=handles, loc="upper left")
ax.set_yticks([f.fold_number for f in folds])
ax.set_yticklabels([f"Fold {f.fold_number}" for f in folds])
ax.set_xlabel("Date")
ax.set_title(f"Rolling-origin fold windows — {demo_name!r}")
ax.xaxis_date()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 5. Candidate Models

Five model classes are registered.  Every function shares the same
interface: `(training_series, horizon) → ModelResult`.  Each wraps its
body in `try/except` so a failing model never aborts the run.

| Model | Complexity | Key property |
|---|---|---|
| `naive` | 0 | Constant = last observed value |
| `seasonal_naive` | 1 | Repeats the last weekly pattern |
| `moving_average` | 2 | Mean of the most recent seasonal window |
| `ets` | 3 | Error–Trend–Seasonal exponential smoothing |
| `auto_arima` | 4 | SARIMA(p,d,q)(P,D,Q)₇ via stepwise search |

Selection tie-breaking uses this complexity order — when two models are
within `RELATIVE_IMPROVEMENT_TOLERANCE` of each other in median MASE,
the simpler model is preferred.

In [ ]:
MODEL_REGISTRY: dict = {
    "naive":          forecast_naive,
    "seasonal_naive": forecast_seasonal_naive,
    "moving_average": forecast_moving_average,
}

if not USE_FAST_MODELS_ONLY:
    MODEL_REGISTRY["ets"]        = forecast_ets
    MODEL_REGISTRY["auto_arima"] = forecast_auto_arima

print(f"Active models ({len(MODEL_REGISTRY)}): {list(MODEL_REGISTRY)}")
if USE_FAST_MODELS_ONLY:
    print("  ↳ Set USE_FAST_MODELS_ONLY = False to include ETS and Auto-ARIMA.")

model_info = pd.DataFrame([
    {"model": "naive",          "complexity": 0, "requires_extra": False,
     "description": "Constant forecast equal to the last observed value."},
    {"model": "seasonal_naive", "complexity": 1, "requires_extra": False,
     "description": "Repeats the last observed 7-day seasonal pattern."},
    {"model": "moving_average", "complexity": 2, "requires_extra": False,
     "description": "Mean of the most recent seasonal window."},
    {"model": "ets",            "complexity": 3, "requires_extra": True,
     "description": "Exponential smoothing (statsmodels ExponentialSmoothing)."},
    {"model": "auto_arima",     "complexity": 4, "requires_extra": True,
     "description": "SARIMA via pmdarima auto_arima with m=7."},
]).assign(active=lambda df: df["model"].isin(MODEL_REGISTRY))

display(model_info)

## 6. Fold-Level Predictions

`evaluate_models_across_folds` fits each registered model on the training
window of every fold, generates a 28-day forecast, and records both the
raw predictions and evaluation metrics.

Evaluation is strictly **out-of-sample**: the model never sees the test
window during fitting.  MASE denominators are computed from the fold's
own training window (not the full series), so each fold is self-contained.

In [ ]:
print(f"Evaluating {len(MODEL_REGISTRY)} model(s) × {len(folds)} fold(s) "
      f"for {demo_name!r}…")

predictions, fold_metrics = evaluate_models_across_folds(
    report_id     = demo_rid,
    series        = demo_series,
    model_registry= MODEL_REGISTRY,
    backtest_config= DEMO_BACKTEST_CONFIG,
)

print(f"  Prediction rows:   {len(predictions):,}")
print(f"  Fold-metric rows:  {len(fold_metrics):,}")

In [ ]:
last_fold  = folds[-1]
fold_preds = predictions[predictions["fold_number"] == last_fold.fold_number].copy()
fold_preds["forecast_date"] = pd.to_datetime(fold_preds["forecast_date"])

# Show 4 weeks of training context immediately before the test window
context_start = last_fold.train_end - pd.Timedelta(days=28)
context = demo_series.loc[context_start:last_fold.train_end]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(context.index, context.values,
        color="dimgray", linewidth=1.5, label="Observed (training context)")

# Reference actuals — same for every model; take from first model in registry
first_model = list(MODEL_REGISTRY)[0]
actuals_df = fold_preds[fold_preds["model_name"] == first_model]
ax.plot(actuals_df["forecast_date"], actuals_df["actual"],
        color="black", linewidth=2.0, linestyle="--", label="Actual (test window)")

# One forecast line per model
_COLOURS = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for i, model_name in enumerate(MODEL_REGISTRY):
    mdf = fold_preds[fold_preds["model_name"] == model_name]
    if mdf.empty or mdf["forecast"].isna().all():
        continue
    ax.plot(mdf["forecast_date"], mdf["forecast"],
            label=model_name, color=_COLOURS[i % len(_COLOURS)],
            linewidth=1.4, alpha=0.85)

ax.axvline(last_fold.cutoff_date, color="gray", linestyle=":", linewidth=1.0,
           label=f"Cutoff (fold {last_fold.fold_number})")
ax.set_title(f"Fold {last_fold.fold_number} — actual vs candidate forecasts\n{demo_name!r}")
ax.set_xlabel("Date")
ax.set_ylabel("Daily views")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

## 7. Model Metrics

Metrics are computed **per fold per model**:

| Metric | Formula | Interpretation |
|---|---|---|
| **MAE** | mean(|actual − forecast|) | Absolute error in original units |
| **WAPE** | Σ|actual − forecast| / Σ|actual| | Scale-free error; NaN when all actuals zero |
| **MASE** | MAE / (seasonal-naive MAE on training window) | < 1 beats seasonal naive |
| **Bias** | mean(forecast − actual) | Positive = over-forecast |
| **Interval coverage** | fraction of actuals within prediction interval | Calibration measure |

MASE < 1 means the model beats seasonal naive over that fold.  Consistent
MASE < 1 across folds is the minimum bar for model selection.

In [ ]:
metric_cols = [
    "fold_number", "model_name", "mae", "rmse", "wape",
    "mase", "bias", "interval_coverage", "fit_status",
]
demo_fold_metrics = fold_metrics[
    [c for c in metric_cols if c in fold_metrics.columns]
].sort_values(["fold_number", "mase"]).reset_index(drop=True)

display(demo_fold_metrics)

## 8. Horizon-Bucket Performance

Errors are decomposed across four horizon sub-windows to reveal how
forecast quality degrades with lead time:

| Bucket | Steps | Description |
|---|---|---|
| `days_1_7` | 1 – 7 | Near-term: within one weekly cycle |
| `days_8_14` | 8 – 14 | Mid-term: second weekly cycle |
| `days_15_28` | 15 – 28 | Long-term: weeks 3-4 |
| `full_horizon` | 1 – 28 | Summary across the whole horizon |

The **MASE denominator** is always the seasonal-naive error on the fold's
own training window — constant per fold so bucket MASE values are
directly comparable.

In [ ]:
bucket_metrics = calculate_horizon_bucket_metrics(
    predictions    = predictions,
    series_lookup  = {demo_rid: demo_series},
    seasonal_period= SEASONAL_PERIOD,
)

bucket_cols = [
    "model_name", "fold_number", "horizon_bucket",
    "mae", "wape", "mase", "bias", "observation_count",
]
display(
    bucket_metrics[[c for c in bucket_cols if c in bucket_metrics.columns]]
    .sort_values(["fold_number", "horizon_bucket", "mase"])
    .reset_index(drop=True)
    .head(60)
)

In [ ]:
bucket_order = ["days_1_7", "days_8_14", "days_15_28", "full_horizon"]
bucket_summary = (
    bucket_metrics
    .groupby(["model_name", "horizon_bucket"])[["mae", "mase"]]
    .mean()
    .reset_index()
)

models_in_plot = bucket_summary["model_name"].unique()
x     = np.arange(len(bucket_order))
width = 0.8 / max(len(models_in_plot), 1)
_COLOURS = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

for col_idx, (metric_key, ax) in enumerate(zip(["mae", "mase"], axes)):
    for m_idx, model_name in enumerate(models_in_plot):
        mdata = bucket_summary[bucket_summary["model_name"] == model_name]
        vals = [
            float(mdata.loc[mdata["horizon_bucket"] == b, metric_key].iloc[0])
            if len(mdata[mdata["horizon_bucket"] == b]) else float("nan")
            for b in bucket_order
        ]
        offset = x + m_idx * width
        ax.bar(offset, vals, width, label=model_name,
               color=_COLOURS[m_idx % len(_COLOURS)], alpha=0.8)
    ax.set_xticks(x + width * (len(models_in_plot) - 1) / 2)
    ax.set_xticklabels(bucket_order, rotation=20, ha="right")
    ax.set_ylabel(metric_key.upper())
    ax.set_title(f"{metric_key.upper()} by horizon bucket (mean across folds)")
    ax.legend(fontsize=8)
    if metric_key == "mase":
        ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8,
                   label="Seasonal naive baseline (MASE = 1)")

plt.suptitle(f"Horizon-bucket performance — {demo_name!r}", y=1.02)
plt.tight_layout()
plt.show()

## 9. Cross-Fold Model Summary

`summarise_model_performance` aggregates fold-level metrics into one row
per `(report_id, model_name)`:

* **`median_mase`** — robust central tendency; used for ranking.
* **`fold_win_rate`** — fraction of eligible folds where this model
  had the lowest MASE (within the tie tolerance).
* **`has_sufficient_folds`** — `True` when `valid_folds ≥ MIN_VALID_FOLDS`.
  Models below this threshold are excluded from selection.

In [ ]:
# Evaluate additional reports for a richer summary table.
# The demo report's fold_metrics are already computed above.
all_fold_metrics_list = [fold_metrics]

additional_ids = [r for r in passing_ids if r != demo_rid][:MAX_REPORTS_FOR_SUMMARY - 1]
if additional_ids:
    print(f"Evaluating {len(additional_ids)} additional report(s)…")

for rid in additional_ids:
    try:
        _, fm = evaluate_models_across_folds(
            rid, series_dict[rid], MODEL_REGISTRY, DEMO_BACKTEST_CONFIG
        )
        all_fold_metrics_list.append(fm)
        print(f"  ✓ {rid}")
    except Exception as exc:
        print(f"  ✗ {rid}: {exc}")

combined_fm = pd.concat(all_fold_metrics_list, ignore_index=True)
model_summary = summarise_model_performance(combined_fm)

print(f"\nSummary: {len(model_summary)} row(s) ({model_summary['report_id'].nunique()} report(s))")

summary_cols = [
    "report_id", "model_name", "valid_folds", "failed_folds",
    "has_sufficient_folds", "median_mase", "mean_wape",
    "mean_bias", "fold_win_count", "fold_win_rate",
]
display(
    model_summary[[c for c in summary_cols if c in model_summary.columns]]
    .head(25)
    .reset_index(drop=True)
)

## 10. Model Selection

`select_models` applies a four-gate policy per report:

1. **Fold sufficiency** — exclude models with < `MIN_VALID_FOLDS` valid folds.
2. **Bias guardrail** — exclude models where |mean_bias| / mean_mae >
   `MAX_BIAS_RATIO` (scale-invariant systematic-error check).
3. **Practical tie group** — rank by median MASE.  All models within
   `RELATIVE_IMPROVEMENT_TOLERANCE` of the best form a tie group.
4. **Simplicity preference** — within the tie group, the model with the
   lowest complexity rank is selected.

When no model survives, `selection_status = "no_reliable_model"` with
an explanatory reason.

In [ ]:
selection = select_models(model_summary)

n_selected = (selection["selection_status"] == "selected").sum()
n_no_model = (selection["selection_status"] == "no_reliable_model").sum()

print(f"Reports evaluated:       {len(selection)}")
print(f"  → selected:            {n_selected}")
print(f"  → no_reliable_model:   {n_no_model}")
print(f"\nSelection policy parameters:")
print(f"  RELATIVE_IMPROVEMENT_TOLERANCE: {RELATIVE_IMPROVEMENT_TOLERANCE:.0%}")
print(f"  MAX_BIAS_RATIO:                 {MAX_BIAS_RATIO}")
print(f"  MIN_VALID_FOLDS:                {MIN_VALID_FOLDS}")

sel_cols = [
    "report_id", "selected_model", "selection_status",
    "valid_folds", "median_mase", "improvement_vs_seasonal_naive_pct",
    "seasonal_naive_median_mase",
]
display(
    selection[[c for c in sel_cols if c in selection.columns]]
    .reset_index(drop=True)
)

In [ ]:
print("Selection reasons\n" + "─" * 60)
for _, row in selection.iterrows():
    status_icon = "✓" if row["selection_status"] == "selected" else "✗"
    print(f"{status_icon} {row['report_id']}: {row['selection_reason']}")

## 11. Full-History Refit and Future Forecast

After selection the **evaluation phase is complete**.  A fresh model is
now fitted on the **complete available history** — no train/test split,
no `model.update` on an evaluation model.

This ensures the production model:
- Uses every available observation as training signal.
- Is entirely independent of the evaluation phase.
- Starts forecasting from the day after the final observed date.

`build_production_forecast` returns one row per `(report_id, horizon_step)`
with full selection lineage: `selection_reason`, `selection_run_id`,
`training_start`, and `training_cutoff`.

In [ ]:
eligible_series = {rid: series_dict[rid] for rid in passing_ids}

production_df = build_production_forecast(
    selection       = selection,
    series_dict     = eligible_series,
    run_id          = RUN_ID,
    generated_at    = RUN_TIMESTAMP,
    horizon         = FORECAST_HORIZON_DAYS,
    selection_run_id= RUN_ID,
)

fc_rows = production_df[production_df["horizon_step"].notna()]
print(f"Production forecast rows:        {len(fc_rows):,}")
print(f"Reports with forecasts:          {fc_rows['report_id'].nunique()}")
print(f"Horizon steps per report:        {fc_rows.groupby('report_id')['horizon_step'].count().unique().tolist()}")
print(f"\nOutput columns:")
for col in PRODUCTION_FORECAST_COLS:
    print(f"  {col}")

display(production_df.head(10))

In [ ]:
demo_prod = (
    production_df[
        (production_df["report_id"] == demo_rid)
        & production_df["horizon_step"].notna()
    ]
    .copy()
)
demo_prod["forecast_date"] = pd.to_datetime(demo_prod["forecast_date"])

if demo_prod.empty:
    print(f"No production forecast for {demo_rid!r}.")
    print("Check selection status above — the report may have no reliable model.")
else:
    # 56 days of observed history as visual context
    context_start = demo_series.index.max() - pd.Timedelta(days=56)
    context = demo_series.loc[context_start:]

    selected_model_name = demo_prod["selected_model"].iloc[0]
    training_start_str  = str(demo_prod["training_start"].iloc[0])[:10]
    training_cutoff_str = str(demo_prod["training_cutoff"].iloc[0])[:10]

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(context.index, context.values,
            color="steelblue", linewidth=1.5, label="Observed history (context)")
    ax.plot(demo_prod["forecast_date"], demo_prod["forecast"],
            color="tomato", linewidth=2.0, linestyle="--",
            label=f"Production forecast — {selected_model_name}")

    has_bounds = (
        "lower_bound" in demo_prod.columns
        and demo_prod["lower_bound"].notna().any()
    )
    if has_bounds:
        ax.fill_between(
            demo_prod["forecast_date"],
            demo_prod["lower_bound"].clip(lower=0),
            demo_prod["upper_bound"].clip(lower=0),
            color="tomato", alpha=0.15, label="Prediction interval",
        )

    ax.axvline(demo_series.index.max(), color="gray", linestyle=":",
               linewidth=1.0, label="Training cutoff")
    ax.set_title(
        f"Production forecast — {demo_name!r}\n"
        f"Model: {selected_model_name}  |  "
        f"Training: {training_start_str} → {training_cutoff_str}"
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("Daily views")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    reason = demo_prod["selection_reason"].iloc[0]
    print(f"\nSelection reason: {reason}")

## 12. Limitations

### Synthetic-data assumptions
When this notebook is run against synthetic data produced by
`generate_synthetic_data.py`, the series are generated with simple
weekly patterns and Gaussian noise.  Real Power BI report usage may
exhibit more complex structures (holiday effects, trend breaks,
multi-modal distributions) that the synthetic generator does not capture.

### Weekly seasonality assumption
All models use a fixed seasonal period of **7 days**.  Reports with
monthly, quarterly, or multi-seasonal patterns will be under-fit.
The `seasonal_naive` model and SARIMA's `m=7` parameter both encode
this assumption.  A report with a dominant monthly cycle may show
MASE > 1 for all models against the day-of-week seasonal naive.

### Sparse-report limitations
Reports with many zero-view days are excluded by the eligibility gates
(`MIN_NONZERO_RATIO`).  For the sparse reports that do pass, zero-view
stretches inflate WAPE and can make MASE unstable when the seasonal-naive
denominator is near zero.

### No exogenous variables
The current framework uses univariate time-series models only.  Features
such as report metadata, user behaviour indicators, or calendar events
(public holidays, fiscal quarter boundaries) are not included as
regressors.  A SARIMAX extension would require extending `candidates.py`
and re-running the evaluation framework.